In [ ]:
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.middleware.cors import CORSMiddleware
import pandas as pd
import io

# Initialize the Web Backend
app = FastAPI(title="OPTIMA Engine API")

# Allow your React dashboard to communicate with this backend
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"], # Change this to your Vercel/React URL in production
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.post("/api/upload-data")
async def process_sales_data(file: UploadFile = File(...)):
    """
    Receives an Excel or CSV file from the React dashboard, 
    cleans it, and prepares it for the OPTIMA models.
    """
    if not file.filename.endswith(('.xlsx', '.csv')):
        raise HTTPException(status_code=400, detail="Only .xlsx or .csv files are allowed.")

    try:
        # 1. Read the uploaded file directly from memory (No local folders needed!)
        contents = await file.read()
        dfs = []

        if file.filename.endswith('.xlsx'):
            # Read all sheets dynamically
            excel_data = pd.read_excel(io.BytesIO(contents), sheet_name=None)
            for sheet_name, df in excel_data.items():
                dfs.append(df)
        else:
            # Handle standard CSV
            df = pd.read_csv(io.BytesIO(contents))
            dfs.append(df)

        # Stitch everything together
        sales_df = pd.concat(dfs, ignore_index=True)

        # 2. Standardize Column Names
        sales_df.columns = (sales_df.columns
                            .str.strip().str.lower()
                            .str.replace(' ', '_')
                            .str.replace(r'\(|\)', '', regex=True))

        sales_df = sales_df.rename(columns={
            'orderdate': 'order_date', 
            'itemdescription': 'item_description', 
            'customerid': 'customer_id', 
            'orderid': 'order_id'
        })

        # 3. Standardize Data Types
        sales_df['order_date'] = pd.to_datetime(sales_df['order_date'], dayfirst=True, errors='coerce')
        sales_df['quantity'] = pd.to_numeric(sales_df['quantity'], errors='coerce')
        sales_df['total'] = pd.to_numeric(sales_df['total'], errors='coerce')

        # 4. Anomaly and Return Filtering
        valid_sales = sales_df[(sales_df['quantity'] > 0) & (sales_df['total'] >= 0)].copy()
        valid_sales = valid_sales.dropna(subset=['item_description', 'order_id', 'order_date'])

        # 5. Output for the Web App
        # Instead of saving to a local CSV, we save it to the server's local storage 
        # (or a database) so the forecasting models can access it.
        valid_sales.to_csv("backend_storage/base_cleaned_sales.csv", index=False)

        # Return a success message back to the React frontend
        return {
            "status": "success",
            "message": "Data successfully ingested and cleaned.",
            "total_rows_processed": len(valid_sales),
            "date_range": f"{valid_sales['order_date'].min().date()} to {valid_sales['order_date'].max().date()}"
        }

    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Error processing file: {str(e)}")

FileNotFoundError: No Excel files found.